\
Pre-entrega 2: Pipeline de extracción de entidades técnicas
Recibe un texto sin procesar (log de error o descripción de arquitectura) y devuelve un objeto validado con Pydantic, usando una cadena LCEL de LangChain con reintentos.
---



1. Instalación de librerías

In [1]:
!pip install -q langchain-core langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 816.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.5/862.5 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 13.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


2. Configuración: API key, proveedor y modelo

In [16]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3.8-flash"

3. Modelos de Gemini disponibles

In [22]:
from google import genai

client = genai.Client()
for m in client.models.list():
    if "flash" in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


4. Configuración de logs

In [23]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logging.getLogger("httpx").setLevel(logging.WARNING)
logger = logging.getLogger("pipeline")

5. Esquema Pydantic (contrato de salida)

In [24]:
from enum import Enum
from pydantic import BaseModel, Field, field_validator, ValidationError


class NivelCriticidad(str, Enum):
    BAJA = "baja"
    MEDIA = "media"
    ALTA = "alta"


class ExtraccionTecnica(BaseModel):
    """Entidades técnicas extraídas de un texto (arquitectura de software o log de error)."""

    tecnologias: list[str] = Field(
        ...,
        min_length=1,
        description="Tecnologías, lenguajes, frameworks, bases de datos o servicios "
                    "mencionados explícitamente en el texto, con su nombre oficial.",
    )
    nivel_de_criticidad: NivelCriticidad = Field(
        ...,
        description="Gravedad del problema o riesgo descripto: baja, media o alta.",
    )
    resumen_tecnico: str = Field(
        ...,
        min_length=20,
        max_length=400,
        description="Resumen técnico de una o dos oraciones, en español.",
    )

    @field_validator("tecnologias")
    @classmethod
    def limpiar_tecnologias(cls, valores: list[str]) -> list[str]:
        """Quita espacios, vacíos y duplicados."""
        limpias, vistas = [], set()
        for valor in valores:
            valor = valor.strip()
            if valor and valor.lower() not in vistas:
                limpias.append(valor)
                vistas.add(valor.lower())
        if not limpias:
            raise ValueError("La lista de tecnologías no puede estar vacía")
        return limpias

6. Prueba de validación del esquema
Tres entradas inválidas que se rechazan antes de llamar al modelo.

In [26]:
dpruebas = {
    "valido": {"tecnologias": ["FastAPI", " Redis ", "redis"], "nivel_de_criticidad": "alta",
               "resumen_tecnico": "API con caché en Redis y cuello de botella en conexiones."},
    "lista_vacia": {"tecnologias": [], "nivel_de_criticidad": "alta",
                    "resumen_tecnico": "Resumen suficientemente largo para pasar."},
    "criticidad_invalida": {"tecnologias": ["MySQL"], "nivel_de_criticidad": "urgente",
                            "resumen_tecnico": "Resumen suficientemente largo para pasar."},
    "resumen_corto": {"tecnologias": ["MySQL"], "nivel_de_criticidad": "baja",
                      "resumen_tecnico": "Corto"},
}

for nombre, datos in pruebas.items():
    try:
        obj = ExtraccionTecnica.model_validate(datos)
        print(f"VA {nombre}: {obj.model_dump(mode='json')}")
    except ValidationError as e:
        print(f"NO VA {nombre}: {e.errors()[0]['msg']}")

VA valido: {'tecnologias': ['FastAPI', 'Redis'], 'nivel_de_criticidad': 'alta', 'resumen_tecnico': 'API con caché en Redis y cuello de botella en conexiones.'}
NO VA lista_vacia: List should have at least 1 item after validation, not 0
NO VA criticidad_invalida: Input should be 'baja', 'media' or 'alta'
NO VA resumen_corto: String should have at least 20 characters


7. Creación del modelo (según LLM_PROVIDER)

In [27]:
def crear_modelo():
    proveedor = os.getenv("LLM_PROVIDER", "gemini").lower()
    temperatura = float(os.getenv("LLM_TEMPERATURE", "0"))
    max_tokens = int(os.getenv("LLM_MAX_TOKENS", "2048"))

    if proveedor == "openai":
        from langchain_openai import ChatOpenAI
        nombre = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
        modelo = ChatOpenAI(model=nombre, temperature=temperatura,
                            max_tokens=max_tokens, max_retries=2)
    elif proveedor == "anthropic":
        from langchain_anthropic import ChatAnthropic
        nombre = os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5")
        # Anthropic solo acepta temperatura entre 0 y 1
        modelo = ChatAnthropic(model=nombre, temperature=min(temperatura, 1.0),
                               max_tokens=max_tokens, max_retries=2)
    elif proveedor == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        nombre = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
        modelo = ChatGoogleGenerativeAI(model=nombre, temperature=temperatura,
                                        max_output_tokens=max_tokens, max_retries=2)
    else:
        raise ValueError(f"LLM_PROVIDER no soportado: {proveedor!r}")

    logger.info("Modelo configurado: %s / %s (max_tokens=%d)", proveedor, nombre, max_tokens)
    return modelo

8. Prompt template

In [28]:
from langchain_core.prompts import ChatPromptTemplate

INSTRUCCIONES_FORMATO = """\
- tecnologias: solo las mencionadas explícitamente en el texto, con su nombre oficial \
(por ejemplo "PostgreSQL", no "postgres"). No inventes tecnologías.
- nivel_de_criticidad: "alta" si hay caída del servicio, pérdida de datos o riesgo de seguridad; \
"media" si hay degradación de rendimiento o errores intermitentes; \
"baja" si es una descripción sin problemas o una mejora menor.
- resumen_tecnico: una o dos oraciones en español, como máximo 300 caracteres."""

PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Sos un analista técnico de software. Tu tarea es extraer entidades técnicas "
     "del texto que te pasa el usuario.\n\nReglas de formato:\n{instrucciones_formato}"),
    ("human", "Texto a analizar:\n<<<\n{texto}\n>>>"),
]).partial(instrucciones_formato=INSTRUCCIONES_FORMATO)

# Ver cómo queda el prompt armado (no llama al modelo)
print(PROMPT.format(texto="Ejemplo: la API en FastAPI se cae al conectar con Redis."))

System: Sos un analista técnico de software. Tu tarea es extraer entidades técnicas del texto que te pasa el usuario.

Reglas de formato:
- tecnologias: solo las mencionadas explícitamente en el texto, con su nombre oficial (por ejemplo "PostgreSQL", no "postgres"). No inventes tecnologías.
- nivel_de_criticidad: "alta" si hay caída del servicio, pérdida de datos o riesgo de seguridad; "media" si hay degradación de rendimiento o errores intermitentes; "baja" si es una descripción sin problemas o una mejora menor.
- resumen_tecnico: una o dos oraciones en español, como máximo 300 caracteres.
Human: Texto a analizar:
<<<
Ejemplo: la API en FastAPI se cae al conectar con Redis.
>>>


9. Validación de la salida (finish_reason y parsing)

In [29]:
from langchain_core.exceptions import OutputParserException


class RespuestaIncompletaError(Exception):
    """El modelo cortó la respuesta por límite de tokens."""


class SalidaInvalidaError(Exception):
    """El modelo devolvió JSON mal formado o que no cumple el esquema."""


def _finish_reason(mensaje) -> str:
    meta = getattr(mensaje, "response_metadata", None) or {}
    return str(meta.get("finish_reason") or meta.get("stop_reason") or "desconocido")


def validar_salida(resultado: dict) -> ExtraccionTecnica:
    """Recibe {'raw', 'parsed', 'parsing_error'} y devuelve el objeto validado o lanza error."""
    motivo = _finish_reason(resultado["raw"])
    logger.info("finish_reason del modelo: %s", motivo)

    # 1) Respuesta cortada por tokens: el objeto puede estar incompleto
    if any(m in motivo.upper() for m in ("LENGTH", "MAX_TOKENS")):
        logger.warning("Respuesta cortada por límite de tokens -> se reintentará")
        raise RespuestaIncompletaError(f"finish_reason={motivo}")

    # 2) JSON mal formado o que no cumple el esquema
    if resultado["parsing_error"] is not None:
        logger.warning("Salida inválida: %s -> se reintentará", resultado["parsing_error"])
        raise SalidaInvalidaError(str(resultado["parsing_error"]))

    parsed = resultado["parsed"]
    if parsed is None:
        logger.warning("El modelo no devolvió datos estructurados -> se reintentará")
        raise SalidaInvalidaError("Respuesta sin datos estructurados")
    if isinstance(parsed, dict):
        parsed = ExtraccionTecnica.model_validate(parsed)

    logger.info("Validación OK: %d tecnologías, criticidad=%s",
                len(parsed.tecnologias), parsed.nivel_de_criticidad.value)
    return parsed

10. Cadena LCEL con reintentos

In [30]:
from langchain_core.runnables import RunnableLambda


def construir_cadena():
    modelo = crear_modelo()
    cadena = (
        PROMPT
        | modelo.with_structured_output(ExtraccionTecnica, include_raw=True)
        | RunnableLambda(validar_salida)
    )
    return cadena.with_retry(
        retry_if_exception_type=(
            SalidaInvalidaError,
            RespuestaIncompletaError,
            OutputParserException,
            ValidationError,
        ),
        wait_exponential_jitter=True,
        stop_after_attempt=3,  # 1 intento + 2 reintentos
    )


cadena = construir_cadena()

23:19:08 | INFO    | pipeline | Modelo configurado: gemini / gemini-3.8-flash (max_tokens=2048)


11. Función asíncrona process_text()

In [31]:
async def process_text(text: str) -> ExtraccionTecnica:
    """Ejecuta el pipeline de forma asíncrona y devuelve un objeto validado."""
    if not text or not text.strip():
        raise ValueError("El texto de entrada está vacío")

    logger.info("Procesando texto (%d caracteres)", len(text))
    try:
        resultado = await cadena.ainvoke({"texto": text})
    except Exception as error:
        logger.error("Falló después de los reintentos: %s: %s", type(error).__name__, error)
        raise
    logger.info("Pipeline terminado correctamente")
    return resultado

12. Prueba con un log de error real

In [32]:
import json

texto = (
    "2026-10-14 03:12:44 ERROR [api-gateway] FastAPI worker timeout: "
    "psycopg2.OperationalError: too many connections for role 'app'. "
    "Redis cache miss rate 92%. Kubernetes reinició el pod 4 veces en 10 minutos."
)

resultado = await process_text(texto)
print(json.dumps(resultado.model_dump(mode="json"), ensure_ascii=False, indent=2))

23:19:13 | INFO    | pipeline | Procesando texto (204 caracteres)
23:19:13 | INFO    | google_genai.models | AFC is enabled with max remote calls: 10.
23:22:25 | INFO    | pipeline | finish_reason del modelo: STOP
23:22:25 | INFO    | pipeline | Validación OK: 4 tecnologías, criticidad=alta
23:22:25 | INFO    | pipeline | Pipeline terminado correctamente


{
  "tecnologias": [
    "FastAPI",
    "psycopg2",
    "Redis",
    "Kubernetes"
  ],
  "nivel_de_criticidad": "alta",
  "resumen_tecnico": "Agotamiento del pool de conexiones mediante psycopg2 y alta tasa de fallos en Redis provocaron timeouts en FastAPI, causando cuatro reinicios del pod en Kubernetes."
}


13. Prueba de estrés: varios textos en paralelo
Un log de error, una descripción de arquitectura y un texto ambiguo, procesados a la vez con asyncio.gather.

In [33]:
import asyncio

textos = {
    "log_error": texto,
    "arquitectura": (
        "El frontend en React consume una API REST hecha con NestJS. Los datos se "
        "guardan en MySQL y las imágenes en un bucket de S3. Todo corre en Docker "
        "y se despliega con GitHub Actions. Por ahora no hay problemas reportados."
    ),
    "ambiguo": (
        "Ayer algo andaba lento, creo que era la base o el caché, no sé. "
        "Hoy parece que anda."
    ),
}

resultados = await asyncio.gather(
    *(process_text(t) for t in textos.values()),
    return_exceptions=True,
)

for nombre, r in zip(textos, resultados):
    print(f"\n=== {nombre} ===")
    if isinstance(r, Exception):
        print(f" Error controlado ({type(r).__name__}): {r}")
    else:
        print(json.dumps(r.model_dump(mode="json"), ensure_ascii=False, indent=2))

23:24:43 | INFO    | pipeline | Procesando texto (204 caracteres)
23:24:43 | INFO    | pipeline | Procesando texto (219 caracteres)
23:24:43 | INFO    | pipeline | Procesando texto (84 caracteres)
23:24:43 | INFO    | google_genai.models | AFC is enabled with max remote calls: 10.
23:24:43 | INFO    | google_genai.models | AFC is enabled with max remote calls: 10.
23:24:43 | INFO    | google_genai.models | AFC is enabled with max remote calls: 10.
23:24:50 | INFO    | google_genai._api_client | Retrying google.genai._api_client.BaseApiClient._async_request_once in 1.03 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}.
23:24:50 | INFO    | google_genai._api_client | Retrying google.genai._api_client.BaseApiClient._async_request_once in 1.78 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 


=== log_error ===
 Error controlado (GoogleAPIError): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

=== arquitectura ===
 Error controlado (GoogleAPIError): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

=== ambiguo ===
{
  "tecnologias": [
    "Base de datos",
    "Caché"
  ],
  "nivel_de_criticidad": "media",
  "resumen_tecnico": "Se reportó una degradación temporal de rendimiento atribuida preliminarmente a la base de datos o al sistema de caché, la cual parece haberse normalizado de forma espontánea."
}


14. Prueba de finish_reason: respuesta cortada por tokens
Se baja el límite a 30 tokens para forzar una respuesta incompleta. El pipeline debe detectarla, reintentar y fallar de forma controlada.

In [34]:
os.environ["LLM_MAX_TOKENS"] = "30"
cadena = construir_cadena()

try:
    await process_text(texto)
except Exception as e:
    print(f"Error controlado ({type(e).__name__}): {e}")
finally:
    # Volver a la configuración normal
    os.environ["LLM_MAX_TOKENS"] = "2048"
    cadena = construir_cadena()

23:27:56 | INFO    | pipeline | Modelo configurado: gemini / gemini-3.8-flash (max_tokens=30)
23:27:56 | INFO    | pipeline | Procesando texto (204 caracteres)
23:27:56 | INFO    | google_genai.models | AFC is enabled with max remote calls: 10.
23:30:20 | INFO    | pipeline | finish_reason del modelo: MAX_TOKENS
23:30:20 | WARNING | pipeline | Respuesta cortada por límite de tokens -> se reintentará
23:30:21 | INFO    | google_genai.models | AFC is enabled with max remote calls: 10.
23:30:26 | INFO    | pipeline | finish_reason del modelo: MAX_TOKENS
23:30:26 | WARNING | pipeline | Respuesta cortada por límite de tokens -> se reintentará
23:30:28 | INFO    | google_genai.models | AFC is enabled with max remote calls: 10.
23:30:32 | INFO    | pipeline | finish_reason del modelo: MAX_TOKENS
23:30:32 | WARNING | pipeline | Respuesta cortada por límite de tokens -> se reintentará
23:30:32 | ERROR   | pipeline | Falló después de los reintentos: RespuestaIncompletaError: finish_reason=MAX_TO

Error controlado (RespuestaIncompletaError): finish_reason=MAX_TOKENS
